# is-differentiable-flag — ex3: global grad_tracking toggle overrides is_differentiable

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `is-differentiable-flag`. Running the final beacon cell reports progress against the `Backprop: is_differentiable flag` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: is_differentiable flag` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`is-differentiable-flag`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "is-differentiable-flag"
DD_SUBTOPIC = "Backprop: is_differentiable flag"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## is_differentiable + grad_tracking_enabled — global toggle wins

Ex1 + ex2 fixed `grad_tracking_enabled=True` and varied `is_differentiable`. The deepening move flips the OTHER gate: with `grad_tracking_enabled=False` (inside a `no_grad()`-style block), no op can produce a grad-tracked output, no matter how the per-op flag is set.

```
requires_grad = (grad_tracking_enabled       # GLOBAL — short-circuits everything
                 AND is_differentiable        # per-op closure flag
                 AND any(input.requires_grad))
```

The three gates are a strict AND. Setting any one to `False` zeros the output's `requires_grad` AND skips Recipe construction — exactly like setting `is_differentiable=False` in ex2.

**Why the global toggle takes priority semantically.** A user writing `with no_grad():` is saying 'no autograd activity at all'. That intent must win over any per-op default. Concretely: a differentiable op like `add` with tracked inputs, called inside `no_grad`, must produce `requires_grad=False` and `recipe=None` — indistinguishable from a non-diff op's output.

**Re-entry behaviour.** Flipping `grad_tracking_enabled` back to `True` restores normal behaviour on the next call — the flag is read FRESH each invocation, not captured at wrap-time (unlike `is_differentiable`).

### Exercise 3 — global grad_tracking toggle overrides is_differentiable

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the three-gate AND for requires_grad to show that grad_tracking_enabled=False forces a False output regardless of is_differentiable=True OR any tracked inputs, and re-enabling restores normal behaviour on the next call (fresh read, not closure-captured).
> Keywords: grad-tracking-enabled, no-grad, global-toggle, three-gate
> ```

**KCs targeted:** `global-toggle-short-circuits-three-gate`, `fresh-read-of-runtime-flag`

Implement `wrap_forward_fn(fwd_fn, is_differentiable=True)` — same signature as ex2 — but the test will exercise the GLOBAL toggle interaction.

Requirements:

1. Unbox MiniTensor inputs to raw arrays.
2. Call `fwd_fn(*raw_args, **kwargs)`.
3. **Three-gate AND** (READ GLOBAL FRESH AT CALL TIME, not at wrap-time): `grad_tracking_enabled` AND `is_differentiable` AND `any-tracked-input`.
4. Box the result; attach `Recipe` only when `requires_grad=True`.

The fresh-read part is critical: the test will flip the global flag mid-program and re-call the SAME wrapped op — the flag must take effect IMMEDIATELY on the next call, not stay at whatever it was when `wrap_forward_fn` ran.

Implementation hint: read the flag via `globals()['grad_tracking_enabled']` or simply reference it by name inside the inner `tensor_func` (Python's name resolution will look it up fresh each call).

Verify:
- Global True + is_diff True + tracked input -> requires_grad=True, recipe is built.
- Global False + is_diff True + tracked input -> requires_grad=False, recipe=None (global short-circuits).
- Global True + is_diff False + tracked input -> requires_grad=False, recipe=None (per-op gate short-circuits — same as ex2).
- Toggle global False -> True between calls: behaviour flips back. Without re-wrapping. The same `add` wrapped fn produces both kinds of outputs depending on the live value.

In [ ]:
def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    """Three-gate wrapper with FRESH read of grad_tracking_enabled."""
    raise NotImplementedError()


def _test_ex3():
    # --- Make sure global is ON for the canonical path ---
    globals()['grad_tracking_enabled'] = True
    add = wrap_forward_fn(t.add)                                  # is_diff=True
    eq  = wrap_forward_fn(t.eq, is_differentiable=False)          # is_diff=False

    a = MiniTensor(t.tensor([1.0, 2.0, 3.0]), requires_grad=True)
    b = MiniTensor(t.tensor([4.0, 5.0, 6.0]), requires_grad=True)

    # --- canonical path: global=T, is_diff=T, tracked input -> requires_grad=T ---
    c = add(a, b)
    assert c.requires_grad is True, 'canonical path must produce requires_grad=True'
    assert c.recipe is not None, 'canonical path must build a Recipe'
    assert c.recipe.func is t.add

    # --- per-op gate alone: is_diff=False blocks the Recipe (this is ex2 territory but baseline) ---
    mask = eq(a, b)
    assert mask.requires_grad is False, 'is_diff=False must block requires_grad'
    assert mask.recipe is None

    # === THE HEADLINE: global toggle short-circuits even is_diff=True ===
    globals()['grad_tracking_enabled'] = False
    c2 = add(a, b)
    assert c2.requires_grad is False, (
        f'with grad_tracking_enabled=False, even is_diff=True ops must produce '
        f'requires_grad=False; got {c2.requires_grad}. Did you capture the flag at wrap-time?'
    )
    assert c2.recipe is None, (
        f'with grad_tracking_enabled=False, no Recipe should be built; got {c2.recipe}'
    )
    # Forward value should still be correct — global toggle does not change MATH.
    assert t.allclose(c2.array, a.array + b.array), 'forward result must still be a+b'

    # === FRESH READ: flip global back on; SAME wrapped `add` resumes normal behaviour ===
    globals()['grad_tracking_enabled'] = True
    c3 = add(a, b)
    assert c3.requires_grad is True, (
        'after toggling global back on, requires_grad must return to True — '
        'the flag is read FRESH each call, not closure-captured at wrap-time'
    )
    assert c3.recipe is not None

    # --- non-tracked inputs always produce non-tracked output, regardless of flags ---
    a_const = MiniTensor(t.tensor([1.0, 2.0, 3.0]), requires_grad=False)
    b_const = MiniTensor(t.tensor([4.0, 5.0, 6.0]), requires_grad=False)
    c4 = add(a_const, b_const)
    assert c4.requires_grad is False, 'no tracked input -> no tracked output'
    assert c4.recipe is None

    # --- all-False matrix: every (global, is_diff, tracked) combo with at least one False -> requires_grad=False ---
    combos_false = [
        (False, True,  True),   # global off
        (True,  False, True),   # per-op off
        (True,  True,  False),  # no tracked input
        (False, False, True),
        (False, True,  False),
        (True,  False, False),
        (False, False, False),
    ]
    for (g, isd, tr) in combos_false:
        globals()['grad_tracking_enabled'] = g
        op = wrap_forward_fn(t.add, is_differentiable=isd)
        x = MiniTensor(t.tensor([1.0]), requires_grad=tr)
        y = MiniTensor(t.tensor([2.0]), requires_grad=tr)
        out = op(x, y)
        assert out.requires_grad is False, (
            f'combo (global={g}, is_diff={isd}, tracked={tr}) must give requires_grad=False, got True'
        )
        assert out.recipe is None, (
            f'combo (global={g}, is_diff={isd}, tracked={tr}) must give recipe=None, got {out.recipe}'
        )

    # --- only all-True triple produces requires_grad=True ---
    globals()['grad_tracking_enabled'] = True
    op = wrap_forward_fn(t.add, is_differentiable=True)
    x = MiniTensor(t.tensor([1.0]), requires_grad=True)
    y = MiniTensor(t.tensor([2.0]), requires_grad=True)
    out = op(x, y)
    assert out.requires_grad is True
    assert out.recipe is not None

    # --- leave global on for the rest of the notebook (good hygiene) ---
    globals()['grad_tracking_enabled'] = True
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        # Unbox MiniTensor inputs.
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        # Three-gate AND. Critical: read grad_tracking_enabled FRESH at call time
        # (via globals()), not at wrap-time.
        global_on = globals().get('grad_tracking_enabled', True)
        any_tracked = any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        requires_grad = bool(global_on and is_differentiable and any_tracked)
        out = MiniTensor(out_arr, requires_grad=requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(func=fwd_fn, args=raw_args, kwargs=kwargs, parents=parents)
        return out
    return tensor_func
```

**The closure trap.** If `wrap_forward_fn` captured `grad_tracking_enabled` at wrap-time (e.g. `g = grad_tracking_enabled` outside the inner fn), it would freeze to whatever value the flag had when wrapping happened. A user calling `wrap_forward_fn(t.add)` at module import (flag=True) and later doing `with no_grad(): ...` (flag flipped to False) would still see grad-tracked outputs. Fresh read fixes this.

**Why not capture `is_differentiable` fresh too?** `is_differentiable` IS a per-op closure flag — it's part of the wrapped function's identity. `add` is differentiable; `eq` is not. That's a wrapping-time fact, fixed by the time the function exists. Only the GLOBAL runtime flag needs the fresh-read treatment.

**`global_on AND ...` short-circuits at the first False.** Python's `and` evaluates left-to-right and stops at the first falsy value. Putting `global_on` first makes the common no-grad case (e.g. inside `with t.no_grad():`) skip the `any_tracked` scan entirely — a tiny optimisation that matches PyTorch's actual implementation.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()